# 02 — Colab runner (GPU training)

This notebook only **launches** code from the repository: all logic lives in `src/brain_tumor/`.

**Before the first run**, add these Colab secrets (key icon in the left bar), with notebook access enabled:

| Secret | Value |
|---|---|
| `DAGSHUB_USER` | your DagsHub username |
| `DAGSHUB_TOKEN` | a DagsHub access token |
| `GITHUB_TOKEN` | *(optional)* only to push tuned parameters back |

Runtime → Change runtime type → **GPU**.

In [ ]:
# ---- Settings ----
GITHUB_USER = "your-github-username"
REPO = "brain-tumor-mlops"
BRANCH = "main"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install it

In [ ]:
import os
if not os.path.exists(REPO):
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!git pull -q
!pip install -q -e .
!git log -1 --oneline

## 2. Credentials (DagsHub = DVC storage + MLflow server)

In [ ]:
from google.colab import userdata

user, token = userdata.get("DAGSHUB_USER"), userdata.get("DAGSHUB_TOKEN")
os.environ.update({
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{user}/{REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": user,
    "MLFLOW_TRACKING_PASSWORD": token,
})
# Credentials go to .dvc/config.local, which is git-ignored.
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {user}
!dvc remote modify origin --local password {token}

## 3. Pull only what training needs (~300 MB)

In [ ]:
!dvc pull data/processed data/splits.csv data/metadata.csv
!python -m brain_tumor.data.validate

## 4. Experiments

Run the cells you need. Every run is logged to MLflow with its parameters, git commit,
data version, curves, confusion matrix, Grad-CAM and checkpoint.

In [ ]:
# 0. Majority-class baseline
!python -m brain_tumor.baseline

In [ ]:
# 1-2. Custom CNN without / with augmentation
!python -m brain_tumor.train train.model=custom_cnn train.augment=false
!python -m brain_tumor.train train.model=custom_cnn

In [ ]:
# 3-4. Pretrained models (two-stage fine-tuning)
!python -m brain_tumor.train train.model=resnet18
!python -m brain_tumor.train train.model=efficientnet_b0

In [ ]:
# 5. Leakage demonstration: same model, image-level split
!python -m brain_tumor.train train.model=resnet18 train.split_column=image_split

## 5. (Optional) Optuna tuning — same budget for every compared model

In [ ]:
!python -m brain_tumor.tune train.model=custom_cnn tune.n_trials=20
!python -m brain_tumor.train train.model=custom_cnn --best-params configs/best_params/custom_cnn.yaml --run-name custom_cnn-tuned

In [ ]:
# Push the tuned parameters back to GitHub (needs the GITHUB_TOKEN secret)
gh = userdata.get("GITHUB_TOKEN")
!git config user.email "colab@users.noreply.github.com" && git config user.name "Colab runner"
!git add configs/best_params && git commit -m "exp: tuned hyperparameters" || echo "nothing to commit"
!git push https://{gh}@github.com/{GITHUB_USER}/{REPO}.git HEAD:{BRANCH}

## 6. (Optional) Bonus: 5-fold patient-level cross-validation

In [ ]:
for fold in range(5):
    !python -m brain_tumor.train train.model=resnet18 train.fold={fold}

## 7. Final model: register it in the MLflow Model Registry

In [ ]:
!python -m brain_tumor.train train.model=resnet18 --register --run-name resnet18-final